# VLM-Anomaly — Full MVTec Sweep (Kaggle P100)

**Purpose:** Run all 15 MVTec AD categories × configured VLM backends zero-shot, then write
per-image JSONL results that can be committed back to the repo.

**Setup:**
1. Fork this notebook on Kaggle.
2. Add your API keys as Kaggle Secrets (`TOGETHER_API_KEY`, `GEMINI_API_KEY`,
   `ANTHROPIC_API_KEY`, `GROQ_API_KEY`).
3. Attach the MVTec AD dataset (search Kaggle datasets: `mvtec-anomaly-detection`).
4. Hit **Run All**.  Expect 2–6 hours for a full sweep.
5. Download `results/` from the output and commit to the repo.

**Idempotency:** Each category's JSONL is written incrementally.  Re-running skips
categories whose output file already exists and is non-empty.

In [ ]:
# ── 1. Install ──────────────────────────────────────────────────────────────
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/sabareeswarans11/VLM-Anomaly.git",
    "httpx", "tenacity", "pyyaml", "pydantic>=2.10", "pydantic-settings>=2.6",
    "structlog", "Pillow", "duckdb", "scikit-learn", "tqdm",
])
print("✓ vlm-anomaly installed")

In [ ]:
# ── 2. API keys from Kaggle Secrets ─────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient  # type: ignore[import]

secrets = UserSecretsClient()

def _get(name: str) -> str | None:
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

os.environ["TOGETHER_API_KEY"]   = _get("TOGETHER_API_KEY") or ""
os.environ["GEMINI_API_KEY"]     = _get("GEMINI_API_KEY") or ""
os.environ["ANTHROPIC_API_KEY"]  = _get("ANTHROPIC_API_KEY") or ""
os.environ["GROQ_API_KEY"]       = _get("GROQ_API_KEY") or ""

active_keys = [k for k in ["TOGETHER_API_KEY","GEMINI_API_KEY","ANTHROPIC_API_KEY","GROQ_API_KEY"]
               if os.environ.get(k)]
print(f"Active API keys: {active_keys or 'none — only mock/groq-free will run'}")

In [ ]:
# ── 3. Paths ─────────────────────────────────────────────────────────────────
from pathlib import Path

MVTEC_ROOT = Path("/kaggle/input/mvtec-anomaly-detection")   # Kaggle dataset mount point
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify dataset is present
assert MVTEC_ROOT.exists(), f"MVTec dataset not found at {MVTEC_ROOT}. Attach it in the sidebar."
categories_on_disk = sorted(p.name for p in MVTEC_ROOT.iterdir() if p.is_dir())
print(f"Found {len(categories_on_disk)} categories: {categories_on_disk}")

In [ ]:
# ── 4. Configure sweep ───────────────────────────────────────────────────────
# Edit this cell to choose which backends and prompts to run.

BACKENDS = [
    "groq",       # free — always runs
    # "together",  # requires TOGETHER_API_KEY
    # "gemini",    # requires GEMINI_API_KEY
    # "anthropic", # requires ANTHROPIC_API_KEY
]

PROMPT_KEY = "generic.detailed"   # generic.simple | generic.detailed | generic.cot
LIMIT_PER_CATEGORY = None         # None = all images; set e.g. 10 for a quick test
BUDGET_USD = 5.0                  # per backend

# Filter to keys that are actually set
def _backend_has_key(name: str) -> bool:
    key_map = {"together":"TOGETHER_API_KEY","gemini":"GEMINI_API_KEY",
               "anthropic":"ANTHROPIC_API_KEY","groq":"GROQ_API_KEY"}
    key = key_map.get(name)
    return not key or bool(os.environ.get(key))

BACKENDS = [b for b in BACKENDS if _backend_has_key(b)]
print(f"Running backends: {BACKENDS}")
print(f"Prompt: {PROMPT_KEY} | Limit: {LIMIT_PER_CATEGORY} | Budget: ${BUDGET_USD}/backend")

In [ ]:
# ── 5. Build helper objects ──────────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level="INFO")

settings = Settings(
    _env_file="/dev/null",       # no .env on Kaggle
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
)
# Point dataset loader to the Kaggle mount
dataset = MVTec(root_dir=MVTEC_ROOT)
prompt_lib = PromptLibrary()

print(f"Dataset root  : {dataset.root_dir}")
print(f"Results dir   : {settings.results_dir}")
print(f"Categories    : {dataset.categories()}")

In [ ]:
# ── 6. Run sweep ─────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
from vlm_anomaly.schemas import ExperimentConfig, EvalResult
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator

def _make_backend(name: str):
    if name == "groq":
        from vlm_anomaly.backends.groq import GroqBackend
        return GroqBackend()
    if name == "together":
        from vlm_anomaly.backends.together import TogetherBackend
        return TogetherBackend()
    if name == "gemini":
        from vlm_anomaly.backends.gemini import GeminiBackend
        return GeminiBackend()
    if name == "anthropic":
        from vlm_anomaly.backends.anthropic_backend import AnthropicBackend
        return AnthropicBackend()
    raise ValueError(name)

all_results: list[EvalResult] = []

for backend_name in BACKENDS:
    print(f"\n{'='*60}")
    print(f"Backend: {backend_name}")
    print(f"{'='*60}")
    backend = _make_backend(backend_name)

    for category in tqdm(dataset.categories(), desc=backend_name):
        # Idempotency: skip if non-empty JSONL already exists for this run
        existing = list(RESULTS_DIR.glob(f"*_{category}.jsonl"))
        if existing and existing[0].stat().st_size > 10:
            print(f"  [skip] {category} — output already exists")
            continue

        config = ExperimentConfig(
            backend=backend_name,
            dataset="mvtec",
            categories=[category],
            prompt=PROMPT_KEY,
            limit=LIMIT_PER_CATEGORY,
            budget_usd=BUDGET_USD,
        )
        evaluator = VLMEvaluator(
            backend=backend,
            dataset=dataset,
            config=config,
            settings=settings,
            prompt_library=prompt_lib,
        )
        results = evaluator.run()
        all_results.extend(results)
        for r in results:
            auroc = f"{r.auroc:.3f}" if r.auroc is not None else "N/A"
            print(f"  {r.category:20s}  AUROC={auroc}  n={r.n_images}  cost=${r.total_cost_usd:.4f}")

print("\nSweep complete.")

In [ ]:
# ── 7. Leaderboard ───────────────────────────────────────────────────────────
from vlm_anomaly.analysis.aggregator import leaderboard

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print("No results yet.")
else:
    display(lb[["model_id","category","n_images","auroc","f1","mean_latency_ms","total_cost_usd"]])

In [ ]:
# ── 8. Export for local commit ───────────────────────────────────────────────
# Zip the results directory so it can be downloaded from Kaggle output.
import shutil

shutil.make_archive("/kaggle/working/vlm_anomaly_results", "zip", RESULTS_DIR)
print("Download  /kaggle/working/vlm_anomaly_results.zip")
print()
print("Then locally:")
print("  unzip vlm_anomaly_results.zip -d results/")
print("  git add results/*.jsonl")
print("  git commit -m 'results: add Kaggle full MVTec sweep'")

## Reproduction note

All JSON/JSONL files in `results/` are committed to the repo.
To regenerate the leaderboard and plots locally:

```bash
python -c "
from vlm_anomaly.analysis.report_generator import generate
generate('results/', 'REPORT.md')
print('Done — see REPORT.md and results/plots/')
"
```